In [2]:
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt
import glob
import os

In [3]:
directory = "test"
percorsi_file = sorted(glob.glob(os.path.join(directory, "*.npy")))

if len(percorsi_file) == 0:
    raise FileNotFoundError(f"Nessun file .npy trovato nella cartella '{directory}'")
elif len(percorsi_file) > 1:
    print(f"Attenzione: trovati {len(percorsi_file)} file .npy, uso il primo: {os.path.basename(percorsi_file[0])}")

f = percorsi_file[0]
nome = os.path.splitext(os.path.basename(f))[0]
imgs = np.load(f)
imgs = imgs / 16   # se immagine è UINT16

print(f"Caricato: '{nome}', dtype={imgs.dtype}, shape={imgs.shape}")

Caricato: 'start_exposure', dtype=float64, shape=(20, 1100, 1600)


In [6]:
# --- Caricamento maschera di riferimento ---
mask_path = os.path.join("reference_mask.png")
reference_mask = cv.imread(mask_path, cv.IMREAD_GRAYSCALE)

if reference_mask is None:
    raise FileNotFoundError(f"Maschera non trovata in: {mask_path}")

print(f"Maschera caricata: dtype={reference_mask.dtype}, shape={reference_mask.shape}")

Maschera caricata: dtype=uint8, shape=(1100, 1600)


In [9]:
# --- Caricamento dati di calibrazione ---
calib_path = os.path.join("calib_data.txt")

if not os.path.isfile(calib_path):
    raise FileNotFoundError(f"File di calibrazione non trovato in: {calib_path}")

calib_data = np.loadtxt(calib_path)

# se il file ha una sola riga, np.loadtxt restituisce un array 1D: lo forzo a 2D
if calib_data.ndim == 1:
    calib_data = calib_data.reshape(1, -1)

ref_zero        = calib_data[:, 0]
std_zero_roi    = calib_data[:, 1]
segnale_max     = calib_data[:, 2]
std_segnale_roi = calib_data[:, 3]

print(f"Dati di calibrazione caricati: {calib_data.shape[0]} righe (ROI)")
print(f"{'ROI':<6} {'ref_zero':>15} {'std_zero':>15} {'segnale_max':>15} {'std_segnale':>15}")
print("─" * 70)
for i, riga in enumerate(calib_data):
    ref_zero, std_zero_roi, segnale_max, std_segnale_roi = riga
    print(f"{i:<6} {ref_zero:>15.3f} {std_zero_roi:>15.3f} {segnale_max:>15.3f} {std_segnale_roi:>15.3f}")

Dati di calibrazione caricati: 16 righe (ROI)
ROI           ref_zero        std_zero     segnale_max     std_segnale
──────────────────────────────────────────────────────────────────────
0           714144.833         835.314    10729560.300       18942.121
1           702069.200         940.752    10253713.033        7415.974
2           696852.300         938.573     9886697.767        7242.589
3           696831.133        1143.253    10384095.933        5834.171
4           695654.967         834.172     9906516.700        6383.852
5           652764.500         755.952     8942029.967        6567.524
6           701290.100         754.812    10389887.467        6431.348
7           692175.367         856.221    10030620.367        7222.410
8           700897.567         711.194     9789565.700        8148.349
9           699868.733         949.188    10192083.300        5875.873
10          683882.933         831.287     9460816.767        6320.853
11          606620.633         